# 第11回: Planning and Reflection①

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session11/session11_planning_reflection_1.ipynb)

これまでのエージェントは、依頼を受けてから ReAct ループを回し、必要な Tool を呼び、答えを返してきた。この形は「聞かれたことに答える」までは強いが、「目的を達成する」には足りない。何を達成すれば終わりなのかが決まっておらず、途中の成果物が十分かを判断する仕組みも、うまく進んでいないときに軌道を変える仕組みもないためである。

今回は、エージェントを目的志向のシステムにする4つの能力を搭載する。

- **Goal Setting**: 何を達成すれば完了なのかを、観測できる条件として定める
- **Planning**: 目的へ至る手順を、実行可能なステップ列へ分解する
- **Reflection**: 出した成果物を自分で批評し、作り直す
- **Monitoring**: 実行結果を成功条件と照らし、続ける・作り直す・終える・人間へ戻すを決める

---

## 0. 環境準備

In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
if IS_COLAB:
    !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
    %cd ai-agent-seminar/session11

%pip install -q -e "."

In [ ]:
# @title APIキーの設定
import getpass
import os
from pathlib import Path

# ローカル実行で .env がある場合はそこから読み込む
if Path('.env').exists():
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except ImportError:
        pass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY を入力する: ')

# 第5節で使う guarded_agent の web_search / fetch_url は、import 時に Tavily を初期化する
if not os.environ.get('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass.getpass('TAVILY_API_KEY を入力する: ')

print('OpenAI APIキー設定完了' if os.environ.get('OPENAI_API_KEY') else 'OpenAI APIキー未設定')
print('Tavily APIキー設定完了' if os.environ.get('TAVILY_API_KEY') else 'Tavily APIキー未設定')

---

## 1. 目的志向のエージェント

複雑な依頼は、1回の応答や1回の Tool 呼び出しでは終わらない。まず何を達成すれば終わりかを定め、そこから中間ステップへ分解し、各ステップの成果物が十分かを判断し、全体が目的へ近づいているかを確かめる必要がある。

これらを1つの大きなプロンプトへ詰め込むこともできるが、そうすると「今どのステップにいるか」「何が未達か」「なぜやり直したか」がすべて会話文のコンテキストから認知・理解・判断をする必要がある。今回の実装では、目的・成功条件・計画・観測・判断を**状態**として持つことで、実行中に扱える制御情報へ変える。

責務を分けると、それぞれの役割は次のようになる。

| 能力 | 役割 | 状態として持つもの |
|---|---|---|
| Goal Setting | 何を達成すれば完了か | 目的、成功条件、制約 |
| Planning | どの順で何をするか | ステップ列、現在の位置 |
| Reflection | この成果物で十分か | 批評、指摘の履歴、改稿回数 |
| Monitoring | 全体として次に何をすべきか | 観測、未達ギャップ、次の遷移 |

Reflection と Monitoring は似ているが、見ている対象が違う。Reflection は**成果物の質**を見て、同じステップをやり直すかどうかを決める。Monitoring は**進行**を見て、次のステップへ進むか、計画を組み直すか、終わるか、人間へ戻すかを決める。この2つを1つのノードに混ぜると、「文章が硬い」といった品質の指摘と「計画が現実と合っていない」という進行の問題が同じ判断へ流れ込み、どちらの理由で止まったのかが追えなくなる。

### 統合ループ

4つの能力は、一度きりの直線的な処理ではない。計画は仮説であり、実行結果によって更新される。

[![](https://mermaid.ink/img/pako:eNp1ks9rFDEUx_-VIScFi_RapCfFiwVpbzUe4szb2cGZZMxmVOgW3J1apsVDVWxB-xPF2kMXBC22W_xnspnZ_he-ZHenLcUc8pL38vm-5OUtEV8EQGY80ojFK7_JpPIezVPu4ZDwIoOWuvWEEp1_1_m57p7gPPi7c7F_TsnT297U1KwXChbjiYdovAVQKuLhvWfy7mz1pVd9XtF5vyw2zPpeuXMw6CPeN8VJ9WsF8VESizudNGYcdR6j4RMN3T3V-arOc51vmWJLd_6YYnV4-LWmLeRoeA0-0g_QZCoS3OHTV3nd_WB6e8ODdzVsGQdLaMTgK-TnR6uJAF693N2u1o4QLtdOh0efanjMWL4t4WXUgrYTvBllvg-panuJ4JESErPMjVaTV14rUOdH9fbQbBR1ojHmpHzBsb7ZtVRX4xJsQdquLDej0PJZzBTSzSxxxR6cnV1sftSdnim-lZvH_8mZpDFYKhAcbDOU22-q312z_9O8X7dtMGKcZv2TlJM7HgllFGBvKZkB7hKQCXMOskSlpShRTUiAoouSgMnnlNjAsmVTxheFSC5xKbKwebnN0gDfcj9ioWT2VIPFLeuHwF59btzVrruX_wGwmi1A?type=png)](https://mermaid.live/edit#pako:eNp1ks9rFDEUx_-VIScFi_RapCfFiwVpbzUe4szb2cGZZMxmVOgW3J1apsVDVWxB-xPF2kMXBC22W_xnspnZ_he-ZHenLcUc8pL38vm-5OUtEV8EQGY80ojFK7_JpPIezVPu4ZDwIoOWuvWEEp1_1_m57p7gPPi7c7F_TsnT297U1KwXChbjiYdovAVQKuLhvWfy7mz1pVd9XtF5vyw2zPpeuXMw6CPeN8VJ9WsF8VESizudNGYcdR6j4RMN3T3V-arOc51vmWJLd_6YYnV4-LWmLeRoeA0-0g_QZCoS3OHTV3nd_WB6e8ODdzVsGQdLaMTgK-TnR6uJAF693N2u1o4QLtdOh0efanjMWL4t4WXUgrYTvBllvg-panuJ4JESErPMjVaTV14rUOdH9fbQbBR1ojHmpHzBsb7ZtVRX4xJsQdquLDej0PJZzBTSzSxxxR6cnV1sftSdnim-lZvH_8mZpDFYKhAcbDOU22-q312z_9O8X7dtMGKcZv2TlJM7HgllFGBvKZkB7hKQCXMOskSlpShRTUiAoouSgMnnlNjAsmVTxheFSC5xKbKwebnN0gDfcj9ioWT2VIPFLeuHwF59btzVrruX_wGwmi1A)

Monitoring は単なるログ記録ではなく、観測した結果をもとに、次の判断をする**制御ノード**。

- **complete**: 成功条件を満たしたので終了する
- **continue**: まだ作業が残っているので次のステップへ進む
- **replan**: 計画が現実と合わないので組み直す
- **escalate**: 判断が曖昧、またはリスクが高いので人間へ戻す

このノートブックでは、第2節で Reflection の内側ループだけを作り、第3節と第4節で Goal Setting・Planning・Monitoring を足して外側ループを閉じ、第5節で実行部を Human-in-the-Loop と Sandbox 付きのエージェントへ差し替える。

### 適用判断

動的な計画は万能ではない。柔軟性と予測可能性はトレードオフの関係にあり、手順が既に分かっている処理では、エージェントに計画させるより固定ワークフローの方が速く、安く、結果も安定する。請求書の定型チェック、決まったデータ変換、既定の承認フローなどがこれにあたる。

判断の軸は単純で、**how を実行中に発見する必要があるか**である。

| 判断基準 | 向く設計 |
|---|---|
| 手順が既知で、毎回同じ順序でよい | 固定ワークフロー |
| how を実行中に発見する必要がある | Planning + Monitoring |
| 成功条件を観測できる | Goal Setting + Monitoring |
| 出力の品質が速度やコストより重要 | Reflection |
| 失敗時の影響が大きい、判断が曖昧 | Human-in-the-Loop |
| 生成されたコマンドやコードを実行する | Sandbox |

Reflection にも同じトレードオフがある。改稿のたびに LLM 呼び出しが増えるため、コストとレイテンシは反復回数に比例して増え、履歴が伸びてコンテキストを圧迫する。品質が速度より重要な場合に限って使う。

---

## 2. Reflection

Reflection は、エージェントが自分の出力を評価し、その評価をもとに改善する設計パターン。単純な連鎖では出力がそのまま次の工程へ流れるが、Reflection は途中にフィードバックループを入れる。

処理は4段階になる。

1. **実行**: 最初の成果物を作る
2. **評価・批評**: 成果物を基準に照らして分析する
3. **改善**: 批評をもとに作り直す
4. **反復**: 満足のいく結果になるか、停止条件に達するまで繰り返す

重要なのは、評価する主体を生成する主体から**論理的に分離する**ことである。同じ役割のまま「見直して」と頼んでも、モデルは自分の出力を妥当だと見なしやすい。プロンプトを分け、別のペルソナ（例: 厳密なコードレビュアー、事実確認担当）を与えると、指摘は具体的になり、見落としも減る。この構成を Producer-Critic と呼ぶ。

[![](https://mermaid.ink/img/pako:eNplkc9KAzEQxl8l5KRg9V6kFz0qiEeNSNxN29DuZslmVWh72AXRYg-K_6FaFFGrqAdBaD34MHGzfQyTVFvQnGYmv2_yTaYGHeYSmAewWGVbThlzARaWkQ_0ETisTKwiKJNPmfRk8irjh_QoHrw9pp2-6p8iuDYJcrkCCDhzI4dwzS79hLMbfKaQHXfU3oGML2Syr-Fh11_YKl2Oi0LLNKau2lmzO8LsjWUcTgV1NDRnA9t5aEDGT9nOvUyaMj6T8Z1q9gbdkz_PDdWmUR1BTjZpSPJAtXbV4bkeRwfZbR_B-sjWPxV2HBKIOmCRMJ-RvrS0WTP6kKxSj5oJ0ritnm_U5fXXx7u16OHtdSoIx4IyP9QCkJseTYN8OAVgiVNXf73gEdGZR7iHbQHWkPWBoCgTjyBdQtDFvIKguWgYbYD9Fca8sZyzqFQep1HgYkHmKS5xbKgiroamTlwqGF_8WbpdfuMbKtnPrg?type=png)](https://mermaid.live/edit#pako:eNplkc9KAzEQxl8l5KRg9V6kFz0qiEeNSNxN29DuZslmVWh72AXRYg-K_6FaFFGrqAdBaD34MHGzfQyTVFvQnGYmv2_yTaYGHeYSmAewWGVbThlzARaWkQ_0ETisTKwiKJNPmfRk8irjh_QoHrw9pp2-6p8iuDYJcrkCCDhzI4dwzS79hLMbfKaQHXfU3oGML2Syr-Fh11_YKl2Oi0LLNKau2lmzO8LsjWUcTgV1NDRnA9t5aEDGT9nOvUyaMj6T8Z1q9gbdkz_PDdWmUR1BTjZpSPJAtXbV4bkeRwfZbR_B-sjWPxV2HBKIOmCRMJ-RvrS0WTP6kKxSj5oJ0ritnm_U5fXXx7u16OHtdSoIx4IyP9QCkJseTYN8OAVgiVNXf73gEdGZR7iHbQHWkPWBoCgTjyBdQtDFvIKguWgYbYD9Fca8sZyzqFQep1HgYkHmKS5xbKgiroamTlwqGF_8WbpdfuMbKtnPrg)

Reflection が効くのは、品質・正確さ・制約への適合が速度やコストより重要な場面である。

| 領域 | 生成するもの | 批評の観点 |
|---|---|---|
| 文章生成 | 記事、告知文、要約 | 流れ、トーン、明確さ、抜けている論点 |
| コード生成 | 関数、スクリプト | バグ、境界条件、テスト結果、可読性 |
| 情報統合 | 長文の要約 | 原文の要点との照合、事実の取りこぼし |
| 計画立案 | 手順、戦略 | 実行可能性、制約違反、依存関係の矛盾 |

### 2.1 Reflectionの実装

階乗を計算する Python 関数の実装

- Critic には**品質チェックリスト**を明示的に渡し、その各項目に照らして判定させる。チェックリストは第3節以降の「成功条件」に相当するもので、Reflection が何を基準に合否を出すのかを外から決めるための仕組みを提供する。
- Critic の戻り値は文章ではなく構造化出力にする。`verdict` が制御に使え、`issues` がそのまま次の改稿の入力になる。
- Critic には、未達項目のうち**最も重要なものを1件だけ**挙げさせる。一度に全部直させるより、どの指摘が成果物のどこを変えたかを追いやすく、レビューの実務にも近い。

In [ ]:
# @title Reflection ループの実装
from typing import Literal

from IPython.display import Image, Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.runnables.graph_mermaid import CurveStyle
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

model_id = 'gpt-5.4-nano' # @param ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna', 'gpt-5.5', 'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.4-nano']
llm = ChatOpenAI(model=model_id, temperature=0.2)

TASK_PROMPT = """`calculate_factorial` という名前の Python 関数を書いてください。
整数 `n` を受け取り、その階乗 (n!) を返します。
"""

# 批評の判定基準。第3節以降の「成功条件」にあたる
QUALITY_CHECKLIST = [
    'docstring に引数、戻り値、送出する例外が書かれている',
    '0 の階乗が 1 になる',
    '負の数には ValueError を送出する',
    'bool は int のサブクラスなので、True / False を拒否する',
    '再帰ではなく反復で実装し、大きな n でも再帰上限に達しない',
    '使用例が doctest 形式で書かれている',
]

PRODUCER_SYSTEM = 'あなたは Python の実装者です。要求を満たすコードだけを返します。説明文は書きません。'

CRITIC_SYSTEM = """あなたは Python に詳しいシニアソフトウェアエンジニアです。
提示されたコードを品質チェックリストの各項目に照らしてレビューしてください。
未達の項目があれば、そのうち最も重要なものを1件だけ issues に入れ、verdict を revise にします。
すべて満たしていれば verdict を accept とし、issues は空にします。"""


class Critique(BaseModel):
    """批評の結果。verdict がループの制御に、issues が次の改稿の入力になる。"""

    verdict: Literal['accept', 'revise'] = Field(
        description='全項目を満たすなら accept、そうでなければ revise。'
    )
    issues: list[str] = Field(description='最も重要な未達項目を1件だけ。accept のときは空。')
    guidance: str = Field(description='次の改稿で何をすべきかの指示。1文。')


# Critic だけ構造化出力にする。Producer の出力は成果物そのものなので素の文字列でよい
critic = llm.with_structured_output(Critique)


def checklist_text() -> str:
    return '\n'.join(f'- {item}' for item in QUALITY_CHECKLIST)


# ここから反復。何回目の改稿か、これまでどんな指摘を受けたか、いつ止めるか。
# これらを関数の引数で持ち回るとノードの責務がすぐ曖昧になるため、グラフの状態として持つ
class ReflectionState(TypedDict, total=False):
    task: str
    draft: str
    critique: Critique
    revision_notes: list[str]  # これまでに受けたすべての指摘（記憶）
    issue_log: list[list[str]]  # サイクルごとの指摘。比較のために残す
    iteration: int
    max_iterations: int
    use_memory: bool


def generate(state: ReflectionState) -> dict:
    """初回は生成、2回目以降は指摘を反映して書き直す。"""
    messages = [SystemMessage(content=PRODUCER_SYSTEM), HumanMessage(content=state['task'])]

    if state.get('draft'):
        if state.get('use_memory', True):
            # 記憶あり: 前回の成果物と、これまでの指摘をすべて渡す
            context = f'前回の成果物:\n{state["draft"]}\n\n'
            context += 'これまでに受けた指摘:\n' + '\n'.join(
                f'- {note}' for note in state.get('revision_notes', [])
            )
        else:
            # 記憶なし: 直近の指摘だけを渡す。前回の成果物も過去の指摘も見えない
            context = '次の指摘を踏まえて書いてください。\n' + '\n'.join(
                f'- {issue}' for issue in state['critique'].issues
            )
        messages.append(HumanMessage(content=context))

    response = llm.invoke(messages)
    return {'draft': str(response.content), 'iteration': state.get('iteration', 0) + 1}


def critique(state: ReflectionState) -> dict:
    """成果物をチェックリストに照らして批評し、指摘を記憶へ積む。"""
    result = critic.invoke([
        SystemMessage(content=CRITIC_SYSTEM),
        HumanMessage(content=(
            f'元の要求:\n{TASK_PROMPT}\n\n'
            f'品質チェックリスト:\n{checklist_text()}\n\n'
            f'レビュー対象:\n{state['draft']}'
        )),
    ])
    return {
        'critique': result,
        'revision_notes': state.get('revision_notes', []) + result.issues,
        'issue_log': state.get('issue_log', []) + [result.issues],
    }


def route_after_critique(state: ReflectionState) -> Literal['generate', 'finish']:
    """批評の verdict を分岐に使う。書き直すか、ここで終わるか。"""
    if state['critique'].verdict == 'accept':
        return 'finish'
    # 上限は付け足しの安全装置ではなく必須の構成要素。批評する側は「もっと良くできる点」を
    # いくらでも挙げられるので、accept が出るまで回す設計にすると止まらない（後の実行で確認する）
    if state.get('iteration', 0) >= state.get('max_iterations', 3):
        return 'finish'
    return 'generate'


reflection_builder = StateGraph(ReflectionState)
reflection_builder.add_node('generate', generate)
reflection_builder.add_node('critique', critique)

reflection_builder.add_edge(START, 'generate')
reflection_builder.add_edge('generate', 'critique')
reflection_builder.add_conditional_edges(
    'critique',
    route_after_critique,
    {'generate': 'generate', 'finish': END},
)

reflection_graph = reflection_builder.compile()

print('Model:', model_id)
display(Image(reflection_graph.get_graph().draw_mermaid_png(curve_style=CurveStyle.NATURAL)))

In [ ]:
# @title 反復ループを実行する
def run_reflection(*, use_memory: bool, max_iterations: int = 3) -> ReflectionState:
    return reflection_graph.invoke(
        {
            'task': TASK_PROMPT,
            'max_iterations': max_iterations,
            'use_memory': use_memory,
            'revision_notes': [],
            'issue_log': [],
        },
        config={'recursion_limit': 30},
    )


with_memory = run_reflection(use_memory=True)

print(f'改稿回数: {with_memory["iteration"]} / 最終判定: {with_memory["critique"].verdict}')
for index, issues in enumerate(with_memory['issue_log'], 1):
    print(f'\n--- {index} 回目の批評 ({len(issues)} 件) ---')
    for issue in issues:
        print('-', issue[:120])

display(Markdown(f'#### 最終成果物\n{with_memory["draft"]}'))

### 注意点

- **批評は尽きない**: 上の実行で見たとおり、Critic は要求を満たした後も「より厳密にできる点」を挙げ続ける。`accept` を待つ設計にすると止まらないため、`max_iterations` が必須になる
- **コストとレイテンシが反復に比例する**: 1サイクルごとに生成と批評で2回の LLM 呼び出しが増える。履歴も伸びるため、コンテキスト長の上限にも近づく
- **自己評価は万能ではない**: 同じモデルが書き手と評価者を兼ねると、見落としも共有される。静的チェックのように機械的に確かめられる観点は、LLM に判定させず自分で確かめる
- **止め方を決める**: 上限に達して終わった場合と、`accept` で終わった場合は意味が違う。どちらで終わったかを状態に残さないと、後段が品質を過信する